# 🌿 Case Study — Leaf Detection with YOLOv8

---

## Welcome to the Final Case Study

This notebook is the **culmination of the entire course**: you will apply everything you have learned — from NumPy arrays to neural networks and CNNs — to a **real-world object detection problem**.

The task is **leaf detection**: given an image of a plant, the model must locate each leaf by drawing a bounding box around it.

---

## What is Object Detection?

Object detection goes **beyond simple classification**. Instead of asking *"what is in this image?"*, it asks *"what is in this image, and exactly where?"*

The output is a list of **bounding boxes**, each with a **class label** and a **confidence score**:

```
  ┌──────────────────────────────────┐
  │                                  │
  │   ┌──────────┐   ┌──────────┐   │
  │   │  leaf    │   │  leaf    │   │
  │   │  0.94    │   │  0.87    │   │
  │   └──────────┘   └──────────┘   │
  │                                  │
  └──────────────────────────────────┘
         Input Image with Predictions
```

---

## What is YOLO?

**YOLO** (You Only Look Once) is a family of real-time object detection models. Unlike two-stage detectors (propose regions → classify), YOLO performs detection in a **single forward pass** — making it extremely fast and practical.

We use **YOLOv8** by [Ultralytics](https://docs.ultralytics.com/), which offers:
- State-of-the-art accuracy on standard benchmarks
- Easy-to-use Python API (`model.train()`, `model.predict()`, `model.val()`)
- Built-in training, validation, and inference pipelines
- Export to `.pt`, `.onnx`, `.tflite`, and more

---

## Notebook Structure

| # | Section | Description |
|---|---------|-------------|
| 1 | Dataset | Clone dataset from GitHub |
| 2 | Setup | Install libraries and import modules |
| 3 | Helper Functions | Utility functions used throughout |
| 4 | Train / Val / Test Split | Organise data into subsets |
| 5 | YOLO Configuration | Create YAML config and folder structure |
| 6 | Pre-trained Model | Load weights and run inference |
| 7 | Export Results | Save predictions to CSV |
| 8 | Optional Training | Train from scratch (advanced) |

---
## 1️⃣ Load Dataset

The dataset contains **images of plants** together with **annotation files** (`.csv`) that record, for each image, the bounding-box coordinates of every visible leaf.

We download it from GitHub using `git clone`. This creates a local copy of the repository with:
- `Leaf/train/` — the training images (`.jpg`)
- `Leaf/train.csv` — bounding-box annotations for each training image
- `Leaf/test/leaf/` — test images for inference
- `Leaf/model/best_float32.tflite` — a pre-trained YOLOv8 model in TFLite format

> **Note:** The `!` prefix in Colab cells runs the command in the system shell, not in Python.

In [ ]:
# Clone the dataset and pre-trained model from GitHub.
# This creates the folder '/content/Leaf-YOLOv8-Tuning/' with all required files.
!git clone https://github.com/Bottins/Leaf-YOLOv8-Tuning.git

---
## 2️⃣ Setup — Install & Import Libraries

Before any code, we install the packages that are not pre-installed in Colab and import all modules we will need.

| Library | Purpose |
|---------|----------|
| `ultralytics` | YOLOv8: training, inference, evaluation, export |
| `squarify` | Treemap-style visualisations |
| `opencv-cv2` | Image loading, colour conversion, drawing boxes |
| `pandas` | Reading and manipulating the CSV annotation file |
| `matplotlib` / `seaborn` | Plotting training curves and results |
| `sklearn` | Splitting the dataset into train / val / test |
| `tensorflow` | Required for the TFLite model backend |

We fix a **random seed** (`42`) so the dataset split is identical across runs, making results reproducible and comparable.

In [ ]:
# ── Install packages not available by default in Colab ────────────────────────
!pip install ultralytics --quiet   # YOLOv8 framework
!pip install squarify --quiet      # Treemap visualisations

# ── Standard library ──────────────────────────────────────────────────────────
import os
import csv
import random
import shutil

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import squarify

# ── Computer Vision ───────────────────────────────────────────────────────────
import cv2

# ── Machine Learning ──────────────────────────────────────────────────────────
import sklearn
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

# ── YOLOv8 ────────────────────────────────────────────────────────────────────
import ultralytics
from ultralytics import YOLO

# ── Reproducibility ───────────────────────────────────────────────────────────
# A fixed seed makes the train/val/test split identical on every run.
random_seed = 42
random.seed(random_seed)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_style('darkgrid')
%matplotlib inline

---
## 3️⃣ Helper Functions

All reusable logic is defined here as documented functions, keeping the main workflow clean.

| Function | Description |
|----------|-------------|
| `csv_to_dataframe` | Load a CSV annotation file into a Pandas DataFrame |
| `calc_percentage` | Compute the % of unique images in a subset vs. the full set |
| `dict_to_yaml` | Convert a Python dict to a YAML-formatted string |
| `create_yaml_data` | Write a YAML config file to disk |
| `convert_bbox_to_yolo` | Convert absolute pixel coords → YOLO normalised format |
| `dataframe_yolofiles` | Write per-image `.txt` YOLO label files from a DataFrame |
| `move_images_to_directory` | Copy images listed in a DataFrame to a target folder |
| `leaf_detect` | Run YOLOv8 inference on one image, return annotated RGB frame |
| `plot_images_with_detections` | Show a grid of images with ground-truth boxes drawn |
| `show_csv_results` | Plot all training/validation metric curves from YOLO's CSV |
| `show_directory_images` | Show a grid of images from a folder |
| `show_metrics` | Bar chart of mAP50, mAP50-95, mAP75 |
| `show_detections` | Run inference on random images and display predicted boxes |
| `show_image` | Display a single image at large size |

### YOLO Bounding Box Format

YOLO stores labels in a **normalised** format (values between 0 and 1) so they are independent of image resolution:

```
  class_id  x_center  y_center  width  height

  Image (W × H pixels)
  ┌──────────────────────┐
  │     ┌──────────┐     │
  │     │ (cx, cy) │     │  →  cx = (x + w/2) / W
  │     │    ·     │  w  │     cy = (y + h/2) / H
  │     └──────────┘     │     nw = w / W
  │          h           │     nh = h / H
  └──────────────────────┘
```

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────

def csv_to_dataframe(csv_filepath, columns_name):
    """Load a CSV annotation file into a Pandas DataFrame."""
    df = pd.read_csv(csv_filepath, names=columns_name, header=0)
    return df


def calc_percentage(df_total, df_part, file_id_column):
    """Return the percentage of unique images that df_part has relative to df_total."""
    part  = len(df_part[file_id_column].unique())
    total = len(df_total[file_id_column].unique())
    return (part / total) * 100


# ── YAML Config ───────────────────────────────────────────────────────────────

def dict_to_yaml(data):
    """Convert a Python dictionary to a YAML-formatted string.
    
    The special entry 'format': 'line_break' inserts a blank line in the output.
    """
    yaml_lines = []
    for key, value in data.items():
        if key == 'format' and value == 'line_break':
            yaml_lines.append('')            # blank line for readability
        elif isinstance(value, list):
            yaml_lines.append(f"{key}: {value}")
        else:
            yaml_lines.append(f"{key}: {value}")
    return '\n'.join(yaml_lines)


def create_yaml_data(yaml_file_name, output_dir, data):
    """Write a YAML configuration file to disk, creating the directory if needed."""
    yaml_content = dict_to_yaml(data)
    output_path  = os.path.join(output_dir, f'{yaml_file_name}.yaml')
    os.makedirs(output_dir, exist_ok=True)
    with open(output_path, 'w') as file:
        file.write(yaml_content)
    print(f'YAML file created: {output_path}')


# ── Bounding Box Conversion ───────────────────────────────────────────────────

def convert_bbox_to_yolo(x, y, width, height, img_width, img_height):
    """Convert a bounding box from absolute pixel coords to YOLO normalised format.
    
    Args:
        x, y (float): Top-left corner of the box (pixels).
        width, height (float): Box dimensions (pixels).
        img_width, img_height (int): Full image dimensions.
    Returns:
        tuple: (x_center, y_center, norm_width, norm_height)  — all in [0, 1]
    """
    x_center = (x + width  / 2) / img_width
    y_center = (y + height / 2) / img_height
    width   /= img_width
    height  /= img_height
    return x_center, y_center, width, height


def dataframe_yolofiles(df, output_dir):
    """Generate one YOLO-format .txt label file per image from a DataFrame.
    
    Each output line: class_id x_center y_center width height  (all normalised).
    With a single class (leaf), class_id is always 0.
    """
    os.makedirs(output_dir, exist_ok=True)
    data = {}

    for _, row in df.iterrows():
        image_id   = row['image_id']
        img_width  = int(row['width'])
        img_height = int(row['height'])
        bbox       = eval(row['bbox'])          # parse '[x, y, w, h]' string → list
        x, y, w, h = bbox

        cx, cy, nw, nh = convert_bbox_to_yolo(x, y, w, h, img_width, img_height)

        if image_id not in data:
            data[image_id] = []
        data[image_id].append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

    for image_id, bboxes in data.items():
        txt_path = os.path.join(output_dir, os.path.splitext(image_id)[0] + '.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(bboxes) + '\n')

    print(f'Label files created in: {output_dir}')


# ── File Management ───────────────────────────────────────────────────────────

def move_images_to_directory(df, root_dir, target_dir):
    """Copy images listed in df['image_id'] from root_dir to target_dir."""
    os.makedirs(target_dir, exist_ok=True)
    for image_id in df['image_id'].unique():
        src = os.path.join(root_dir, image_id)
        dst = os.path.join(target_dir, image_id)
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f"Warning: '{image_id}' not found at {src}")
    print(f'Images copied to: {target_dir}')


# ── Inference & Visualisation ─────────────────────────────────────────────────

def leaf_detect(img_path, model):
    """Run YOLOv8 inference on one image; return annotated RGB frame.
    
    OpenCV loads images in BGR order; we convert to RGB for matplotlib.
    """
    img          = cv2.imread(img_path)
    result       = model(img)
    annotated    = result[0].plot()                              # draws boxes + labels
    return cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)


def plot_images_with_detections(images_dir, labels_dir, num_images=16,
                                 grid_size=(4, 4), figsize=(16, 16)):
    """Display a grid of training images with ground-truth bounding boxes.
    
    Use this to visually verify that images and label files are aligned
    before starting training.
    """
    image_files = sorted(os.listdir(images_dir))[:num_images]
    fig, axs = plt.subplots(grid_size[0], grid_size[1], figsize=figsize)

    for i, image_file in enumerate(image_files):
        r, c  = i // grid_size[1], i % grid_size[1]
        image = cv2.imread(os.path.join(images_dir, image_file))

        label_path = os.path.join(labels_dir, os.path.splitext(image_file)[0] + '.txt')
        with open(label_path) as f:
            labels = f.read().strip().split('\n')

        for label in labels:
            parts = label.split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = map(float, parts)
            H, W = image.shape[:2]
            x1 = int((cx - bw/2) * W);  y1 = int((cy - bh/2) * H)
            x2 = int((cx + bw/2) * W);  y2 = int((cy + bh/2) * H)
            cv2.rectangle(image, (x1, y1), (x2, y2), (250, 221, 47), 3)

        axs[r, c].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axs[r, c].axis('off')

    plt.suptitle('Training Samples — Ground Truth Bounding Boxes', fontsize=16)
    plt.tight_layout()
    plt.show()


def show_csv_results(csv_path):
    """Plot all training and validation metrics from YOLO's results.csv.
    
    Loss metrics:
      box_loss — bounding-box coordinate regression error
      cls_loss — classification error
      dfl_loss — Distribution Focal Loss (improves box boundary sharpness)
    
    Evaluation metrics:
      precision — of all predicted boxes, fraction that are correct (no FP)
      recall    — of all real objects, fraction that were detected (no FN)
      mAP50     — mean Average Precision at IoU threshold 0.50
      mAP50-95  — mAP averaged over IoU 0.50:0.95 (strict COCO metric)
    """
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axs = plt.subplots(5, 2, figsize=(15, 15))
    plots = [
        ('train/box_loss',       'Train Box Loss',          (0,0)),
        ('train/cls_loss',       'Train Class Loss',        (0,1)),
        ('train/dfl_loss',       'Train DFL Loss',          (1,0)),
        ('metrics/precision(B)', 'Precision',               (1,1)),
        ('metrics/recall(B)',    'Recall',                  (2,0)),
        ('metrics/mAP50(B)',     'mAP@0.50',                (2,1)),
        ('metrics/mAP50-95(B)',  'mAP@0.50:0.95',           (3,0)),
        ('val/box_loss',         'Validation Box Loss',     (3,1)),
        ('val/cls_loss',         'Validation Class Loss',   (4,0)),
        ('val/dfl_loss',         'Validation DFL Loss',     (4,1)),
    ]
    for col, title, (r, c) in plots:
        sns.lineplot(x='epoch', y=col, data=df, ax=axs[r, c])
        axs[r, c].set(title=title)

    plt.suptitle('Training Metrics and Loss Curves', fontsize=20, y=1.01)
    plt.tight_layout()
    plt.show()


def show_directory_images(directory, num_images, rows=3, columns=3):
    """Show a grid of images from a folder."""
    files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    fig   = plt.figure(figsize=(12, 12))
    for i in range(min(num_images, len(files))):
        img = cv2.imread(os.path.join(directory, files[i]))
        ax  = fig.add_subplot(rows, columns, i + 1)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis('off')
    for j in range(num_images, rows * columns):
        fig.add_subplot(rows, columns, j + 1).axis('off')
    plt.suptitle('Sample Inference Results', fontsize=14)
    plt.tight_layout()
    plt.show()


def show_metrics(metrics):
    """Bar chart of mAP50-95, mAP50, mAP75 with value annotations."""
    ax = sns.barplot(
        x=['mAP50-95', 'mAP50', 'mAP75'],
        y=[metrics.box.map, metrics.box.map50, metrics.box.map75]
    )
    ax.set(title='YOLO Evaluation Metrics — Test Set',
           xlabel='Metric', ylabel='Value', ylim=(0, 1))
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}',
                    (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=12)
    plt.gcf().set_size_inches(8, 6)
    plt.show()


def show_detections(images_dir, model, num_images=16, rows=4, columns=4):
    """Run inference on random images and show the predicted bounding boxes."""
    files    = os.listdir(images_dir)
    selected = random.sample(files, num_images)
    fig, axes = plt.subplots(rows, columns, figsize=(16, 16))
    for i, img_file in enumerate(selected):
        annotated = leaf_detect(os.path.join(images_dir, img_file), model)
        axes[i // columns, i % columns].imshow(annotated)
        axes[i // columns, i % columns].axis('off')
    plt.suptitle('Model Predictions — Bounding Boxes on Test Images', fontsize=14)
    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    plt.show()


def show_image(img_path):
    """Display a single image at large size."""
    img = mpimg.imread(img_path)
    fig, ax = plt.subplots(figsize=(15, 15))
    ax.imshow(img)
    ax.axis('off')
    plt.show()

---
## 4️⃣ Train / Validation / Test Split

We divide the dataset into three non-overlapping subsets:

| Split | Size | Purpose |
|-------|------|---------|
| **Train** | 80% | The model sees these images and learns from them |
| **Validation** | 10% | Used *during* training to monitor generalisation and tune hyperparameters |
| **Test** | 10% | Held out completely — used only *once* to report final performance |

This separation is critical: the **test set must never influence any training decision**, otherwise the final evaluation will be overly optimistic.

```
  Full Dataset (100%)
  ├── Train       (80%)  ──  Model learns weights
  ├── Validation  (10%)  ──  Hyperparameter tuning / early stopping
  └── Test        (10%)  ──  Final unbiased evaluation
```

The split is performed **at the image level** — all bounding boxes for a given image always belong to the same subset.

In [ ]:
# ── Load annotation CSV ───────────────────────────────────────────────────────
# One row per bounding box: image_id, width, height, bbox ('[x, y, w, h]')

columns_name = ['image_id', 'width', 'height', 'bbox']
csv_filepath = '/content/Leaf-YOLOv8-Tuning/Leaf/train.csv'
df           = csv_to_dataframe(csv_filepath, columns_name)

print(f'Total annotation rows : {len(df)}')
print(f'Unique images         : {df["image_id"].nunique()}')
df.head()

# ── Split at image level ──────────────────────────────────────────────────────
# We split the list of unique filenames and then filter rows.
# This guarantees that no image appears in two different splits.

unique_filenames = df['image_id'].unique()

# Step 1: 80% train | 20% remainder
train_files, val_test_files = train_test_split(
    unique_filenames, test_size=0.2, random_state=random_seed
)

# Step 2: split remainder 50/50 → 10% val | 10% test
val_files, test_files = train_test_split(
    val_test_files, test_size=0.5, random_state=random_seed
)

train_df = df[df['image_id'].isin(train_files)]
val_df   = df[df['image_id'].isin(val_files)]
test_df  = df[df['image_id'].isin(test_files)]

print(f'\nSplit summary:')
print(f'  Train      : {train_df["image_id"].nunique():4d} images  '
      f'({calc_percentage(df, train_df, "image_id"):.1f}%)')
print(f'  Validation : {val_df["image_id"].nunique():4d} images  '
      f'({calc_percentage(df, val_df, "image_id"):.1f}%)')
print(f'  Test       : {test_df["image_id"].nunique():4d} images  '
      f'({calc_percentage(df, test_df, "image_id"):.1f}%)')

---
## 5️⃣ YOLO Configuration

YOLOv8 expects data in a **specific folder structure** and reads a **YAML configuration file** that points to the images and class definitions.

### Required Folder Structure

```
yolo_dataset_leafdetection/
├── data.yaml              ← YOLO config: paths + class names
├── train/
│   ├── images/            ← training images (.jpg)
│   └── labels/            ← YOLO .txt annotation files
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

### The `data.yaml` File

The configuration file tells YOLO:
- Where the image directories are for each split
- `nc: 1` — number of classes (just `leaf`)
- `names: ['leaf']` — the name for class index 0

After creating the folder structure, we visualise 16 training samples to **verify** that images and label files are correctly aligned before any training starts.

In [ ]:
# ── Root output directory ─────────────────────────────────────────────────────
output_dir = '/content/working/yolo_dataset_leafdetection'

# ── Create data.yaml ──────────────────────────────────────────────────────────
yaml_data = {
    'train':  f'{output_dir}/train/images',
    'val':    f'{output_dir}/valid/images',
    'test':   f'{output_dir}/test/images',
    'format': 'line_break',     # inserts blank line before nc/names
    'nc':     1,
    'names':  ['leaf']
}
create_yaml_data('data', output_dir, yaml_data)

# ── Populate each split folder ────────────────────────────────────────────────
# For every split we need to:
#   1. copy the corresponding .jpg files  → split/images/
#   2. generate YOLO-format .txt labels   → split/labels/

src_images = '/content/Leaf-YOLOv8-Tuning/Leaf/train/'

for split_name, split_df in [('train', train_df), ('valid', val_df), ('test', test_df)]:
    move_images_to_directory(split_df, src_images, f'{output_dir}/{split_name}/images')
    dataframe_yolofiles(split_df, f'{output_dir}/{split_name}/labels')

# ── Sanity check: visualise 16 training samples with ground-truth boxes ───────
# If boxes appear in the right positions the conversion pipeline is correct.

print('\nVerification — 16 training images with ground-truth bounding boxes:')
plot_images_with_detections(
    f'{output_dir}/train/images',
    f'{output_dir}/train/labels'
)

# ── Check a sample image resolution ──────────────────────────────────────────
# Knowing the native resolution helps choose the right imgsz for training/inference.
sample = cv2.imread(f'{output_dir}/train/images/LEAF_1171.jpg')
h, w, c = sample.shape
print(f'Sample image: {w} × {h} px — {c} channels')

---
## 6️⃣ Load Pre-trained Model & Run Inference

Instead of training from scratch, we load a **pre-trained YOLOv8 model** that was already fine-tuned on this leaf dataset. The weights are stored in **TFLite format** (`best_float32.tflite`), a compact format designed for deployment on mobile and edge devices.

### Inference Parameters

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `imgsz` | `1024` | Resize images to 1024×1024 before inference (must match training resolution) |
| `conf` | `0.2` | Confidence threshold — only keep predictions with score ≥ 0.2 |
| `save` | `True` | Write annotated output images to `/content/runs/detect/predict/` |
| `save_txt` | `True` | Write predicted labels as YOLO `.txt` files (needed for CSV export) |

> **Confidence threshold:** Lowering it (e.g. 0.1) catches more objects but produces more false positives. Raising it (e.g. 0.5) is more conservative. Tune this based on your tolerance for false alarms vs. missed detections.

In [ ]:
# ── Load the pre-trained TFLite model ─────────────────────────────────────────
# YOLO() accepts .pt, .onnx, .tflite and other formats transparently.
model = YOLO('/content/Leaf-YOLOv8-Tuning/Leaf/model/best_float32.tflite')

# ── Remove any previous prediction folder ─────────────────────────────────────
# Without this, YOLO would create 'predict2/', 'predict3/', … on repeated runs.
predict_dir = '/content/runs/detect/predict'
if os.path.exists(predict_dir):
    shutil.rmtree(predict_dir)
    print('Cleared previous prediction folder.')

# ── Run batch inference on all test images ────────────────────────────────────
# model.predict() processes every image in the given source folder.
model.predict(
    source   = '/content/Leaf-YOLOv8-Tuning/Leaf/test/leaf',
    save     = True,       # save annotated images
    save_txt = True,       # save .txt label files (for CSV export in section 7)
    imgsz    = 1024,       # input resolution (must match training imgsz)
    conf     = 0.2         # minimum confidence to keep a detection
)

# ── Display a sample of results ───────────────────────────────────────────────
print('\nSample inference results:')
show_directory_images(predict_dir, num_images=7, rows=3, columns=3)

---
## 7️⃣ Export Predictions to CSV

After inference we parse the `.txt` output files and produce a **structured CSV summary** with one row per detected image.

For each image we compute:

| Column | Description |
|--------|-------------|
| `image_id` | Filename (without extension) |
| `numero_rilevazioni` | Number of leaf instances detected |
| `altezza` | Vertical extent of all detections combined (pixels) |
| `larghezza` | Horizontal extent of all detections combined (pixels) |

Bounding boxes are first converted back from YOLO's normalised format to **absolute pixel coordinates** using the original image dimensions.

The **union bounding box** (the smallest rectangle enclosing *all* detections) is then used to compute the overall height and width. This gives a rough measure of how much of the image is covered by leaves.

In [ ]:
from PIL import Image

# ── Paths ─────────────────────────────────────────────────────────────────────
image_dir  = '/content/Leaf-YOLOv8-Tuning/Leaf/test/leaf'  # original test images
label_dir  = '/content/runs/detect/predict/labels'          # YOLO .txt predictions
output_csv = '/content/detection_summary.csv'


def get_image_dimensions(path):
    """Return (width, height) without fully decoding the image."""
    with Image.open(path) as img:
        return img.width, img.height


summary_data = []

for label_file in os.listdir(label_dir):
    image_id   = label_file.replace('.txt', '')
    image_path = os.path.join(image_dir, f'{image_id}.jpg')

    if not os.path.exists(image_path):
        continue

    img_w, img_h = get_image_dimensions(image_path)

    # ── Parse YOLO predictions → absolute pixel coordinates ───────────────────
    bboxes = []
    with open(os.path.join(label_dir, label_file)) as f:
        for line in f:
            parts = line.strip().split()
            cx, cy, bw, bh = map(float, parts[1:])
            xmin = (cx - bw/2) * img_w
            ymin = (cy - bh/2) * img_h
            xmax = (cx + bw/2) * img_w
            ymax = (cy + bh/2) * img_h
            bboxes.append((xmin, ymin, xmax, ymax))

    if not bboxes:
        continue

    # ── Union bounding box: enclosing rectangle of all detections ─────────────
    xmin_all = min(b[0] for b in bboxes)
    ymin_all = min(b[1] for b in bboxes)
    xmax_all = max(b[2] for b in bboxes)
    ymax_all = max(b[3] for b in bboxes)

    H = round(ymax_all - ymin_all, 2)
    W = round(xmax_all - xmin_all, 2)

    summary_data.append([image_id, len(bboxes), H, W])

# ── Save CSV ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(
    summary_data,
    columns=['image_id', 'numero_rilevazioni', 'altezza', 'larghezza']
)
results_df.to_csv(output_csv, index=False)

print(f'CSV saved to : {output_csv}')
print(f'Images with detections: {len(results_df)}')
results_df.describe()

---
## 8️⃣ Optional — Train from Scratch

The sections below let you **train the model independently** — useful if you want to experiment with different hyperparameters, a larger dataset, or a custom set of object classes.

### Transfer Learning

We start from `yolov8n.pt` (nano variant, pre-trained on COCO), which has already learned rich general visual features. We then **fine-tune** its weights on the leaf dataset. This is **transfer learning**: instead of learning from random weights, the model starts from a strong base and adapts quickly with far less data.

```
  Pre-trained on COCO               Fine-tuned on Leaves
  (80 classes, 118k images)         (1 class, our dataset)
  ┌──────────────────────┐           ┌───────────────────┐
  │ Backbone (features)  │  ──────►  │ Backbone (frozen) │
  │ Neck (FPN)           │           │ Neck (fine-tuned)  │
  │ Head (COCO classes)  │           │ Head (1 class)     │
  └──────────────────────┘           └───────────────────┘
```

### Key Hyperparameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| `epochs` | `20` | Passes through the training set (increase to 100+ for better accuracy) |
| `imgsz` | `128` | Training resolution — lower is faster; use 640 or 1024 for production |
| `batch` | `8` | Images per gradient update |
| `workers` | `4` | CPU threads for data loading |
| `seed` | `42` | Ensures reproducible weight initialisation and augmentation |

> **GPU tip:** For meaningful results, run this section with a GPU runtime:  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Load YOLOv8 nano backbone (auto-downloaded on first run) ──────────────────
model = YOLO('yolov8n.pt')

# ── Fine-tune on the leaf dataset ─────────────────────────────────────────────
# model.train() handles: data loading, augmentation, optimisation,
# learning-rate scheduling, and saving the best checkpoint automatically.
model.train(
    data    = f'{output_dir}/data.yaml',
    epochs  = 20,
    imgsz   = 128,
    seed    = random_seed,
    batch   = 8,
    workers = 4
)

# Training outputs saved to /content/runs/detect/train/
#   best.pt       ← best model checkpoint (highest val mAP)
#   last.pt       ← checkpoint at the final epoch
#   results.csv   ← per-epoch metrics (used in 8b)

### 8a — Evaluate on the Test Set

`model.val()` computes standard object detection metrics on the split you specify.

**Key metrics:**
- **Precision** — of all boxes the model predicted, what fraction were actually correct? *(guards against false positives)*
- **Recall** — of all real objects in the images, what fraction did the model find? *(guards against missed detections)*
- **mAP50** — mean Average Precision at IoU ≥ 0.50. A box is a true positive if it overlaps the ground truth by at least 50%.
- **mAP50-95** — mAP averaged over IoU thresholds 0.50, 0.55, …, 0.95. This is the official COCO metric and rewards accurate localisation.

In [ ]:
# ── Evaluate on the held-out test split ───────────────────────────────────────
# conf=0.25: only count detections with confidence >= 25% as positive.
# split='test': use the 'test' path from data.yaml.
metrics = model.val(conf=0.25, split='test')

print(f"\n{'─'*38}")
print(f"  mAP@0.50       : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95  : {metrics.box.map:.4f}")
print(f"  mAP@0.75       : {metrics.box.map75:.4f}")
print(f"{'─'*38}\n")

show_metrics(metrics)

### 8b — Training Loss & Metric Curves

These plots show how the model evolved over each epoch. Signs of **healthy training**:
- Training **and** validation losses both decrease steadily
- Precision, recall, and mAP all increase
- Train and validation curves track each other closely (a large gap signals overfitting)

In [ ]:
# ── Plot training curves ───────────────────────────────────────────────────────
# YOLO appends one row per epoch to results.csv during training.
csv_path = '/content/runs/detect/train/results.csv'
show_csv_results(csv_path)

### 8c — Visual Detections on Test Images

We run inference on random test images and display the predicted bounding boxes. This gives an intuitive sense of what the model actually learned — where it succeeds and where it still makes mistakes.

In [ ]:
# ── Show predictions on 16 random test images ─────────────────────────────────
show_detections(
    images_dir = f'{output_dir}/test/images',
    model      = model,
    num_images = 16,
    rows       = 4,
    columns    = 4
)

In [ ]:
# ── (Optional) Export the trained model to TFLite ─────────────────────────────
# Uncomment to produce a .tflite file suitable for mobile / edge deployment.

# model.export(format='tflite')

---
## 🎉 Course Recap

Congratulations — you have completed the full course!

| # | Notebook | Key Concepts |
|---|----------|--------------|
| 01 | Python Basics | Variables, loops, functions, data structures |
| 02 | NumPy & Linear Algebra | Arrays, matrix operations, vectorisation |
| 03 | Visualisation & Interpolation | Matplotlib, curve fitting, signal processing |
| 04 | Learning from Data | Regression, classification, scikit-learn |
| 05 | The Artificial Neuron | Perceptron, activation functions, gradient descent |
| 06 | Neural Networks & MLP | Backpropagation, hidden layers, overfitting |
| 07 | CNNs with CIFAR-10 | Convolutions, pooling, image classification |
| **08** | **YoloLeaf — Case Study** | **Object detection, YOLOv8, bounding boxes, fine-tuning** |

---

### What's Next?

- **Multi-class detection** — extend the dataset with disease or species labels
- **Instance segmentation** — use `YOLOv8-seg` to output pixel-level masks instead of boxes
- **Deployment** — export to TFLite or ONNX and run on a Raspberry Pi or smartphone
- **Data augmentation** — experiment with Albumentations or YOLO's built-in pipeline to improve generalisation on small datasets